# 🏋️ Challenge 04 — Functions, Closures, Decorators & Generators
*Covers Modules 08–09.* 🟢 easy · 🟡 medium · 🔴 hard.

In [1]:
def check(name, test):
    try:
        ok = bool(test())
    except Exception as e:
        ok = False
        name += f"   (raised {type(e).__name__}: {e})"
    print("✅ PASS" if ok else "❌ FAIL", "-", name)

### 🟢 Problem 1 — Flexible arguments
Write `describe(name, *hobbies, **details)` that returns a string:
`describe("Asha", "chess", "code", age=19, city="Pune")` → `"Asha likes chess, code | age=19, city=Pune"`.
With no hobbies use `"nothing"` and with no details omit the ` | ...` part.

In [2]:
def describe(name, *hobbies, **details):
    text = f"{name} likes {', '.join(hobbies) if hobbies else 'nothing'}"
    if details:
        text += " | " + ", ".join(f"{k}={v}" for k, v in details.items())
    return text

In [3]:
check("full", lambda: describe("Asha", "chess", "code", age=19, city="Pune") == "Asha likes chess, code | age=19, city=Pune")
check("no hobbies", lambda: describe("Ravi", age=20) == "Ravi likes nothing | age=20")
check("nothing extra", lambda: describe("Tom") == "Tom likes nothing")

✅ PASS - full
✅ PASS - no hobbies
✅ PASS - nothing extra


### 🟢 Problem 2 — Fix the mutable-default bug
`add_task(task, tasks=...)` should append `task` to `tasks` and return it; when no list is given it must start a **fresh** list every call (so two separate calls never share state).

In [4]:
def add_task(task, tasks=None):
    if tasks is None:
        tasks = []
    tasks.append(task)
    return tasks

In [5]:
check("first call", lambda: add_task("a") == ["a"])
check("second call is independent", lambda: add_task("b") == ["b"])
check("uses given list", lambda: add_task("c", ["x"]) == ["x", "c"])

✅ PASS - first call
✅ PASS - second call is independent
✅ PASS - uses given list


### 🟢 Problem 3 — Higher-order helpers
Write `apply_n(f, n, x)` that applies function `f` to `x` **n times** (`apply_n(lambda v: v*2, 3, 1)` → 8) and `compose(*funcs)` that returns a function applying the given functions **right to left** (`compose(f, g)(x) == f(g(x))`).

In [6]:
def apply_n(f, n, x):
    for _ in range(n):
        x = f(x)
    return x

def compose(*funcs):
    def run(x):
        for f in reversed(funcs):
            x = f(x)
        return x
    return run

In [7]:
check("apply_n doubling", lambda: apply_n(lambda v: v * 2, 3, 1) == 8)
check("apply_n zero times", lambda: apply_n(str.upper, 0, "a") == "a")
check("compose order", lambda: compose(lambda x: x + 1, lambda x: x * 2)(5) == 11)
check("compose 3 funcs", lambda: compose(str, abs, int)("-7") == "7")

✅ PASS - apply_n doubling
✅ PASS - apply_n zero times
✅ PASS - compose order
✅ PASS - compose 3 funcs


### 🟡 Problem 4 — Closure counter & accumulator
`make_counter(start=0, step=1)` returns a function; each call returns the current value then advances it.
`make_averager()` returns a function that takes a number and returns the running average of all numbers seen so far.

<details><summary>💡 Hint</summary>Use `nonlocal` for the counter; for the averager keep `total` and `count`, or a list in the enclosing scope.</details>

In [8]:
def make_counter(start=0, step=1):
    value = start
    def counter():
        nonlocal value
        current = value
        value += step
        return current
    return counter

def make_averager():
    total = count = 0
    def add(x):
        nonlocal total, count
        total += x
        count += 1
        return total / count
    return add

In [9]:
def _c():
    c = make_counter(10, 5); return [c(), c(), c()]
check("counter", lambda: _c() == [10, 15, 20])
def _two():
    a, b = make_counter(), make_counter(); a(); a(); return b()
check("independent counters", lambda: _two() == 0)
def _avg():
    avg = make_averager(); return [avg(10), avg(20), avg(60)]
check("averager", lambda: _avg() == [10.0, 15.0, 30.0])

✅ PASS - counter
✅ PASS - independent counters
✅ PASS - averager


### 🟡 Problem 5 — `map`/`filter`/`reduce` without loops
Write `sum_of_squares_of_evens(nums)` using `filter`, `map` and `sum`, and `product(nums)` using `functools.reduce` (empty list → `1`).

In [10]:
from functools import reduce

def sum_of_squares_of_evens(nums):
    return sum(map(lambda x: x * x, filter(lambda x: x % 2 == 0, nums)))

def product(nums):
    return reduce(lambda a, b: a * b, nums, 1)

In [11]:
check("squares of evens", lambda: sum_of_squares_of_evens([1, 2, 3, 4]) == 20)
check("no evens", lambda: sum_of_squares_of_evens([1, 3]) == 0)
check("product", lambda: product([1, 2, 3, 4]) == 24)
check("empty product", lambda: product([]) == 1)

✅ PASS - squares of evens
✅ PASS - no evens
✅ PASS - product
✅ PASS - empty product


### 🟡 Problem 6 — A `timer` decorator that records call counts
Write a decorator `count_calls` that adds an attribute `.calls` (starting at 0) to the decorated function and increments it every call. It must preserve the function's name (`functools.wraps`).

In [12]:
import functools

def count_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

In [13]:
@count_calls
def hello(name):
    """Say hello."""
    return f"hi {name}"
hello("a"); hello("b")
check("counts calls", lambda: hello.calls == 2)
check("returns value", lambda: hello("z") == "hi z")
check("keeps name and doc", lambda: hello.__name__ == "hello" and hello.__doc__ == "Say hello.")

✅ PASS - counts calls
✅ PASS - returns value
✅ PASS - keeps name and doc


### 🟡 Problem 7 — Decorator with arguments: `retry`
`retry(times)` decorates a function so that if it raises an exception it is retried up to `times` **total attempts**; if every attempt fails, the **last exception is re-raised**. On success return the function's result.

In [14]:
import functools

def retry(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last = None
            for _ in range(times):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last = e
            raise last
        return wrapper
    return decorator

In [15]:
state = {"n": 0}
@retry(3)
def flaky():
    state["n"] += 1
    if state["n"] < 3:
        raise ValueError("not yet")
    return "ok"
check("succeeds on 3rd attempt", lambda: flaky() == "ok" and state["n"] == 3)

@retry(2)
def always_fails():
    raise KeyError("boom")
def _raises():
    try:
        always_fails()
    except KeyError:
        return True
    return False
check("re-raises last exception", _raises)

✅ PASS - succeeds on 3rd attempt
✅ PASS - re-raises last exception


### 🟡 Problem 8 — Memoise it
Write your own decorator `memoize` (a dict cache keyed by the positional arguments) so that `fib(80)` returns instantly, and expose a `.cache` attribute holding the dict.

In [16]:
import functools

def memoize(func):
    cache = {}
    @functools.wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    wrapper.cache = cache
    return wrapper

In [17]:
@memoize
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
check("fib(80)", lambda: fib(80) == 23416728348467685)
check("cache is populated", lambda: len(fib.cache) == 81)

✅ PASS - fib(80)
✅ PASS - cache is populated


### 🟡 Problem 9 — Generators
Write generator functions:
- `evens()` — infinite generator of 0, 2, 4, …
- `take(n, iterable)` — yields the first `n` items of any iterable
- `chunks(seq, size)` — yields consecutive slices of length `size` (last may be shorter)

In [18]:
def evens():
    n = 0
    while True:
        yield n
        n += 2

def take(n, iterable):
    for i, item in enumerate(iterable):
        if i >= n:
            return
        yield item

def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

In [19]:
check("take from infinite", lambda: list(take(5, evens())) == [0, 2, 4, 6, 8])
check("take more than available", lambda: list(take(10, [1, 2])) == [1, 2])
check("take zero", lambda: list(take(0, evens())) == [])
check("chunks", lambda: list(chunks([1, 2, 3, 4, 5], 2)) == [[1, 2], [3, 4], [5]])
check("chunks of string", lambda: list(chunks("abcdef", 4)) == ["abcd", "ef"])

✅ PASS - take from infinite
✅ PASS - take more than available
✅ PASS - take zero
✅ PASS - chunks
✅ PASS - chunks of string


### 🔴 Problem 10 — A generator pipeline
Given a multi-line log string, write generators (lazy!) and chain them:
- `parse(lines)` yields `(level, message)` from lines like `"ERROR: disk full"`
- `only(level, records)` yields the records with that level
- `messages(records)` yields just the messages

Then `errors_in(log)` returns `list(messages(only("ERROR", parse(log.splitlines()))))`.

In [20]:
def parse(lines):
    for line in lines:
        if ":" in line:
            level, _, msg = line.partition(":")
            yield level.strip(), msg.strip()

def only(level, records):
    for lvl, msg in records:
        if lvl == level:
            yield lvl, msg

def messages(records):
    for _, msg in records:
        yield msg

def errors_in(log):
    return list(messages(only("ERROR", parse(log.splitlines()))))

In [21]:
LOG = """INFO: started
ERROR: disk full
WARN: slow
ERROR: timeout: retrying
garbage line"""
check("errors_in", lambda: errors_in(LOG) == ["disk full", "timeout: retrying"])
check("parse is lazy (generator)", lambda: hasattr(parse([]), "__next__"))
check("skips lines without colon", lambda: list(parse(["nope"])) == [])

✅ PASS - errors_in
✅ PASS - parse is lazy (generator)
✅ PASS - skips lines without colon


### 🔴 Problem 11 — Custom iterator class
Write a class `Countdown(n)` that is an **iterator**: iterating yields `n, n-1, ..., 1`. It should be **re-iterable** (a second `for` loop starts again from `n`), so `__iter__` must return a *fresh* iterator each time (hint: make `__iter__` a generator method).

In [22]:
class Countdown:
    def __init__(self, n):
        self.n = n

    def __iter__(self):
        current = self.n
        while current > 0:
            yield current
            current -= 1

In [23]:
cd = Countdown(3)
check("first pass", lambda: list(cd) == [3, 2, 1])
check("second pass works too", lambda: list(cd) == [3, 2, 1])
check("works with sum/max", lambda: sum(Countdown(4)) == 10 and max(Countdown(4)) == 4)
check("zero", lambda: list(Countdown(0)) == [])

✅ PASS - first pass
✅ PASS - second pass works too
✅ PASS - works with sum/max
✅ PASS - zero


### 🔴 Problem 12 — Context manager
Write `class Indent` — a context manager that increases an indentation level by 2 spaces while inside the `with` block. It has method `.say(text)` returning the text prefixed with the current indentation. Nested `with Indent` blocks compound. Use a shared class-level counter or pass the parent.

Easier version: write it as a class with `__enter__` returning `self` after incrementing `Indent.level` and `__exit__` decrementing it.

In [24]:
class Indent:
    level = 0

    def __enter__(self):
        Indent.level += 1
        return self

    def __exit__(self, exc_type, exc, tb):
        Indent.level -= 1
        return False          # do not swallow exceptions

    @staticmethod
    def say(text):
        return "  " * Indent.level + text

In [25]:
check("no indent outside", lambda: Indent.say("a") == "a")
def _nested():
    out = []
    with Indent():
        out.append(Indent.say("b"))
        with Indent():
            out.append(Indent.say("c"))
        out.append(Indent.say("d"))
    out.append(Indent.say("e"))
    return out
check("nested levels", lambda: _nested() == ["  b", "    c", "  d", "e"])
def _exc():
    try:
        with Indent():
            raise ValueError
    except ValueError:
        return Indent.level == 0
check("level restored after exception", _exc)

✅ PASS - no indent outside
✅ PASS - nested levels
✅ PASS - level restored after exception


---
## 🏁 Next
≥ 90 %? Move on to `05_Files_Exceptions_challenges.ipynb`. **Bonus:** write a `@timed` decorator that prints how long each call took, and a `@validate_types` decorator that reads type hints and raises `TypeError` on wrong argument types.